# Сводная статистика по портфелю

Ноутбук собирает статистику в разрезе:

**Сегмент → Зона проблемности → Тип операции (Актив / УО)**

и суммирует поле **«Задолженность тыс. BYN»** по 7 критериям.

> Если исходный файл у вас в формате `.numbers`, сначала экспортируйте его из Apple Numbers в `.xlsx`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)


## 1. Настройки

Укажите путь к исходному Excel-файлу и путь для сохранения результата.

In [ ]:
# Исходный Excel-файл
INPUT_FILE = Path('Портфель.xlsx')

# Итоговый Excel-файл
OUTPUT_FILE = Path('Статистика_портфеля.xlsx')

# 0 = первый лист Excel-файла
SHEET_NAME = 0


## 2. Вспомогательные функции

In [ ]:
def normalize_column_name(x):
    """Нормализация названия колонки для более устойчивого поиска."""
    return (
        str(x)
        .replace('\\n', ' ')
        .replace('\\r', ' ')
        .strip()
        .lower()
        .replace('ё', 'е')
    )


def find_column(df, variants):
    """Ищет колонку по одному из возможных вариантов названия."""
    normalized_columns = {
        normalize_column_name(col): col
        for col in df.columns
    }

    # Точное совпадение
    for variant in variants:
        variant_norm = normalize_column_name(variant)
        if variant_norm in normalized_columns:
            return normalized_columns[variant_norm]

    # Частичное совпадение
    for variant in variants:
        variant_norm = normalize_column_name(variant)
        for normalized_col, original_col in normalized_columns.items():
            if variant_norm in normalized_col:
                return original_col

    raise KeyError(
        f'Не найдена колонка. Искомые варианты: {variants}\\n'
        f'Колонки в файле: {list(df.columns)}'
    )


def prepare_flag(series):
    """Приводит флаг к 0/1. Поддерживает 0/1, Да/Нет, True/False."""
    s = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .str.replace(',', '.', regex=False)
    )

    mapping = {
        '1': 1, '1.0': 1, 'да': 1, 'yes': 1, 'true': 1,
        '0': 0, '0.0': 0, 'нет': 0, 'no': 0, 'false': 0,
        'nan': 0, 'none': 0, '': 0,
    }

    result = s.map(mapping)
    numeric = pd.to_numeric(s, errors='coerce')
    return result.fillna(numeric)


def prepare_number(series):
    """Приводит задолженность к числу, включая значения вида '1 234,56'."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce').fillna(0)

    s = (
        series.astype(str)
        .str.replace('\\xa0', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
    )
    return pd.to_numeric(s, errors='coerce').fillna(0)


## 3. Чтение исходного файла

In [ ]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME)

print(f'Строк в исходном файле: {len(df):,}')
print('\\nКолонки в файле:')
display(pd.DataFrame({'Колонка': df.columns}))

display(df.head())


## 4. Поиск нужных колонок

In [ ]:
COL_SEGMENT = find_column(df, ['Сегмент'])
COL_ZONE = find_column(df, ['Зона проблемности', 'Зона'])
COL_OPERATION = find_column(df, ['Тип операции', 'Операция'])
COL_DEBT = find_column(df, [
    'Задолженность тыс. BYN',
    'Задолженность, тыс. BYN',
    'Задолженность'
])
COL_NI = find_column(df, ['НИ'])
COL_PFN = find_column(df, ['ПФН'])
COL_RESTRA = find_column(df, ['Рестра', 'Реструктуризация', 'Реструкт'])
COL_COLLATERAL = find_column(df, ['Обеспеченность'])

columns_used = pd.DataFrame({
    'Поле': [
        'Сегмент', 'Зона проблемности', 'Тип операции',
        'Задолженность', 'НИ', 'ПФН', 'Рестра', 'Обеспеченность'
    ],
    'Найденная колонка': [
        COL_SEGMENT, COL_ZONE, COL_OPERATION, COL_DEBT,
        COL_NI, COL_PFN, COL_RESTRA, COL_COLLATERAL
    ]
})

display(columns_used)


## 5. Подготовка данных

In [ ]:
df['_НИ'] = prepare_flag(df[COL_NI])
df['_ПФН'] = prepare_flag(df[COL_PFN])
df['_Рестра'] = prepare_flag(df[COL_RESTRA])
df['_Задолженность'] = prepare_number(df[COL_DEBT])

collateral = (
    df[COL_COLLATERAL]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
    .str.replace('ё', 'е')
)

# Подхватывает: Необеспеченный / Необеспеченная / Необеспеченной и т.п.
df['_Необеспеченный'] = collateral.str.contains('необеспеч', na=False)
df['_Обеспеченный'] = ~df['_Необеспеченный']


## 6. Формирование 7 критериев

Критерии сделаны **взаимоисключающими**, чтобы одна и та же задолженность не была посчитана дважды.

Приоритет:

1. `Рестра = 1` → критерий 7
2. затем `ПФН = 1` → критерии 5–6
3. затем `НИ = 1` → критерии 3–4
4. остальные → критерии 1–2

In [ ]:
C1 = '1. НИ=0; ПФН=0; Рестра=0; обеспеченный'
C2 = '2. НИ=0; ПФН=0; Рестра=0; необеспеченный'
C3 = '3. НИ=1; ПФН=0; Рестра=0; обеспеченный'
C4 = '4. НИ=1; ПФН=0; Рестра=0; необеспеченный'
C5 = '5. ПФН=1; обеспеченный'
C6 = '6. ПФН=1; необеспеченный'
C7 = '7. Рестра=1'

conditions = [
    # 7. Рестра = 1
    df['_Рестра'].eq(1),

    # 5. ПФН = 1, обеспеченный
    df['_Рестра'].eq(0) & df['_ПФН'].eq(1) & df['_Обеспеченный'],

    # 6. ПФН = 1, необеспеченный
    df['_Рестра'].eq(0) & df['_ПФН'].eq(1) & df['_Необеспеченный'],

    # 3. НИ = 1, ПФН = 0, рестра = 0, обеспеченный
    df['_Рестра'].eq(0) & df['_ПФН'].eq(0) & df['_НИ'].eq(1) & df['_Обеспеченный'],

    # 4. НИ = 1, ПФН = 0, рестра = 0, необеспеченный
    df['_Рестра'].eq(0) & df['_ПФН'].eq(0) & df['_НИ'].eq(1) & df['_Необеспеченный'],

    # 1. НИ = 0, ПФН = 0, рестра = 0, обеспеченный
    df['_Рестра'].eq(0) & df['_ПФН'].eq(0) & df['_НИ'].eq(0) & df['_Обеспеченный'],

    # 2. НИ = 0, ПФН = 0, рестра = 0, необеспеченный
    df['_Рестра'].eq(0) & df['_ПФН'].eq(0) & df['_НИ'].eq(0) & df['_Необеспеченный'],
]

choices = [C7, C5, C6, C3, C4, C1, C2]

df['Критерий'] = np.select(
    conditions,
    choices,
    default='НЕ РАСПРЕДЕЛЕНО'
)

display(df['Критерий'].value_counts(dropna=False).to_frame('Количество строк'))


## 7. Оставляем операции «Актив» и «УО»

In [ ]:
operation_normalized = (
    df[COL_OPERATION]
    .fillna('')
    .astype(str)
    .str.strip()
    .str.lower()
)

df_result = df[
    operation_normalized.isin(['актив', 'уо'])
].copy()

print(f'Строк после отбора Актив/УО: {len(df_result):,}')


## 8. Построение сводной таблицы

In [ ]:
criteria_order = [C1, C2, C3, C4, C5, C6, C7]

summary = (
    df_result[df_result['Критерий'] != 'НЕ РАСПРЕДЕЛЕНО']
    .pivot_table(
        index=[COL_SEGMENT, COL_ZONE, COL_OPERATION],
        columns='Критерий',
        values='_Задолженность',
        aggfunc='sum',
        fill_value=0,
    )
    .reindex(columns=criteria_order, fill_value=0)
    .reset_index()
)

summary['ИТОГО'] = summary[criteria_order].sum(axis=1)

display(summary.head(20))


## 9. Контроль сумм

Этот блок помогает проверить, что задолженность не потерялась.

In [ ]:
control = pd.DataFrame({
    'Показатель': [
        'Количество исходных строк',
        'Общая задолженность исходного файла',
        'Задолженность Актив + УО',
        'Распределено по 7 критериям',
        'Не распределено',
    ],
    'Значение': [
        len(df),
        df['_Задолженность'].sum(),
        df_result['_Задолженность'].sum(),
        df_result.loc[
            df_result['Критерий'] != 'НЕ РАСПРЕДЕЛЕНО',
            '_Задолженность'
        ].sum(),
        df_result.loc[
            df_result['Критерий'] == 'НЕ РАСПРЕДЕЛЕНО',
            '_Задолженность'
        ].sum(),
    ]
})

unclassified = df_result[
    df_result['Критерий'] == 'НЕ РАСПРЕДЕЛЕНО'
].copy()

display(control)


## 10. Сохранение результата в Excel

Файл будет содержать три листа:

- **Свод** — основная таблица;
- **Контроль** — проверка сумм;
- **Не распределено** — строки, которые не попали ни в один критерий.

In [ ]:
with pd.ExcelWriter(OUTPUT_FILE, engine='xlsxwriter') as writer:
    summary.to_excel(writer, sheet_name='Свод', index=False)
    control.to_excel(writer, sheet_name='Контроль', index=False)
    unclassified.to_excel(writer, sheet_name='Не распределено', index=False)

    workbook = writer.book
    ws = writer.sheets['Свод']

    header_format = workbook.add_format({
        'bold': True,
        'text_wrap': True,
        'valign': 'top',
        'align': 'center',
        'border': 1,
        'bg_color': '#D9EAF7',
    })

    money_format = workbook.add_format({
        'num_format': '#,##0.00',
        'border': 1,
    })

    text_format = workbook.add_format({'border': 1})

    for col_num, value in enumerate(summary.columns):
        ws.write(0, col_num, value, header_format)

    ws.set_column(0, 0, 20, text_format)
    ws.set_column(1, 1, 22, text_format)
    ws.set_column(2, 2, 15, text_format)
    ws.set_column(3, len(summary.columns) - 1, 22, money_format)

    ws.freeze_panes(1, 3)
    ws.autofilter(0, 0, len(summary), len(summary.columns) - 1)

print(f'Готово: {OUTPUT_FILE.resolve()}')
